# Strict CNN-Transformer v2 Improved Recalculation Notebook — fixed loader

This notebook recalculates the CNN-Transformer thesis results using the **actual trained CNN-Transformer model**.

This fixed version handles the `AttentionPooling` loading issue by trying the direct-weight implementation used by the saved model. It only continues if the loaded model has **613,889 total parameters**.


In [7]:
# Install dependencies
!pip -q install librosa scikit-learn pandas matplotlib tensorflow requests


In [8]:
import os
import glob
import json
import zipfile
import requests
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import tensorflow as tf

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)

print("TensorFlow:", tf.__version__)


TensorFlow: 2.20.0


In [9]:
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Configuration

In [10]:
# =========================
# Core paths
# =========================
DRIVE_ROOT = Path("/content/drive/MyDrive")
PROCESSED_DIR = DRIVE_ROOT / "drone_audio_processed"

OUTPUT_DIR = PROCESSED_DIR / "cnn_transformer_v2_improved_recalculated_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# This is the model that should have 613,889 total parameters.
MODEL_CANDIDATES = [
    PROCESSED_DIR / "model_cnntransformer_v2_improved.keras",
    DRIVE_ROOT / "model_cnntransformer_v2_improved.keras",
]

EXPECTED_TOTAL_PARAMS = 613_889
EXPECTED_TRAINABLE_PARAMS = 612_865
EXPECTED_NON_TRAINABLE_PARAMS = 1_024

# Internal waveform cache created by your DADS preprocessing pipeline.
WAVEFORM_CACHE_CANDIDATES = [
    PROCESSED_DIR / "waveform_0.50s_hop0.25s_recordsplit.npz",
    DRIVE_ROOT / "waveform_0.50s_hop0.25s_recordsplit.npz",
]

# Audio / feature settings.
SR_TARGET = 16_000
WIN_S = 0.50
HOP_S = 0.25
N_MELS = 64
N_FFT = 512
HOP_LENGTH = 128
FMAX = 8_000
TOP_DB = 80

# Use the same value as the trained model preprocessing.
# If unsure, the notebook can run both variants below and compare them.
PREPROCESSING_VARIANTS = {
    "no_preemphasis": 0.0,
    "preemphasis_0p97": 0.97,
}

# Thresholds used for thesis threshold sweep.
THRESHOLDS = np.array([0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.75, 0.95], dtype=float)

BATCH_SIZE = 128
EXTERNAL_AGGREGATION = "max"   # clip probability = max window probability

print("Output directory:", OUTPUT_DIR)
print("\nModel candidates:")
for p in MODEL_CANDIDATES:
    print(" ", p.exists(), p)

print("\nWaveform cache candidates:")
for p in WAVEFORM_CACHE_CANDIDATES:
    print(" ", p.exists(), p)


Output directory: /content/drive/MyDrive/drone_audio_processed/cnn_transformer_v2_improved_recalculated_results

Model candidates:
  True /content/drive/MyDrive/drone_audio_processed/model_cnntransformer_v2_improved.keras
  False /content/drive/MyDrive/model_cnntransformer_v2_improved.keras

Waveform cache candidates:
  True /content/drive/MyDrive/drone_audio_processed/waveform_0.50s_hop0.25s_recordsplit.npz
  False /content/drive/MyDrive/waveform_0.50s_hop0.25s_recordsplit.npz


## 2. Load the exact CNN-Transformer v2 improved model

In [11]:
# ---- Custom AttentionPooling variants for model loading ----
# The real CNN-Transformer v2 improved model has 613,889 parameters.
# It was saved with a custom layer named AttentionPooling.
# Some saved versions store the attention weights directly on the layer,
# while others store them inside a nested Dense layer.  This notebook tries
# both implementations and only continues if the loaded model has exactly
# 613,889 parameters.

class AttentionPoolingDirect(tf.keras.layers.Layer):
    """Attention pooling with weights stored directly on the layer.

    This fixes the Keras error:
    Layer 'score' expected 2 variables, but received 0 variables.

    That error happens when the saved model has the attention variables on the
    custom layer itself, but the notebook recreates it as a nested Dense layer.
    """
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        feature_dim = int(input_shape[-1])
        self.kernel = self.add_weight(
            name="kernel",
            shape=(feature_dim, 1),
            initializer="glorot_uniform",
            trainable=True,
        )
        self.bias = self.add_weight(
            name="bias",
            shape=(1,),
            initializer="zeros",
            trainable=True,
        )
        super().build(input_shape)

    def call(self, inputs, mask=None, training=None):
        scores = tf.matmul(inputs, self.kernel) + self.bias
        if mask is not None:
            mask = tf.cast(mask[:, :, None], scores.dtype)
            scores = tf.where(mask > 0, scores, tf.constant(-1e9, dtype=scores.dtype))
        weights = tf.nn.softmax(scores, axis=1)
        weights = tf.cast(weights, inputs.dtype)
        return tf.reduce_sum(inputs * weights, axis=1)

    def get_config(self):
        return super().get_config()


class AttentionPoolingDenseScore(tf.keras.layers.Layer):
    """Attention pooling with a nested Dense(1) score layer."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.score = tf.keras.layers.Dense(1, name="score")

    def call(self, inputs, mask=None, training=None):
        scores = self.score(inputs)
        if mask is not None:
            mask = tf.cast(mask[:, :, None], scores.dtype)
            scores = tf.where(mask > 0, scores, tf.constant(-1e9, dtype=scores.dtype))
        weights = tf.nn.softmax(scores, axis=1)
        weights = tf.cast(weights, inputs.dtype)
        return tf.reduce_sum(inputs * weights, axis=1)

    def get_config(self):
        return super().get_config()


class AttentionPoolingMeanFallback(tf.keras.layers.Layer):
    """Last-resort no-weight pooling. Used only for diagnostics, not thesis results."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, inputs, mask=None, training=None):
        return tf.reduce_mean(inputs, axis=1)

    def get_config(self):
        return super().get_config()


ATTENTION_POOLING_VARIANTS = [
    ("direct_weights", AttentionPoolingDirect),
    ("nested_dense_score", AttentionPoolingDenseScore),
    ("mean_fallback_diagnostic_only", AttentionPoolingMeanFallback),
]


In [12]:
def find_existing(paths):
    for p in paths:
        if p.exists():
            return p
    raise FileNotFoundError("None of these paths exist:\n" + "\n".join(str(p) for p in paths))


def load_model_with_attentionpooling_variants(model_path):
    errors = []
    for variant_name, layer_cls in ATTENTION_POOLING_VARIANTS:
        print("\nTrying AttentionPooling variant:", variant_name)
        custom_objects = {"AttentionPooling": layer_cls}
        try:
            with tf.keras.utils.custom_object_scope(custom_objects):
                loaded = tf.keras.models.load_model(
                    model_path,
                    compile=False,
                    safe_mode=False,
                    custom_objects=custom_objects,
                )
            n_params = loaded.count_params()
            print(f"Loaded with {variant_name}. Total params: {n_params:,}")

            # Only accept the true 613,889-parameter CNN-Transformer.
            if n_params == EXPECTED_TOTAL_PARAMS:
                print("PASS: Parameter count matches the real CNN-Transformer v2 improved model.")
                return loaded, variant_name
            else:
                errors.append(f"{variant_name}: loaded but wrong parameter count {n_params:,}")
        except Exception as e:
            msg = f"{variant_name}: {type(e).__name__}: {str(e)[:800]}"
            print("FAILED:", msg)
            errors.append(msg)

    raise RuntimeError(
        "Could not load the 613,889-parameter CNN-Transformer model with any AttentionPooling variant.\n\n"
        + "\n\n".join(errors)
    )


MODEL_PATH = find_existing(MODEL_CANDIDATES)
print("Loading model:", MODEL_PATH)

model, attentionpooling_variant_used = load_model_with_attentionpooling_variants(MODEL_PATH)

total_params = model.count_params()
trainable_params = int(np.sum([np.prod(v.shape) for v in model.trainable_weights]))
non_trainable_params = int(np.sum([np.prod(v.shape) for v in model.non_trainable_weights]))

print("\nModel path:", MODEL_PATH)
print("AttentionPooling variant used:", attentionpooling_variant_used)
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)
print("Total params:", total_params)
print("Trainable params:", trainable_params)
print("Non-trainable params:", non_trainable_params)

if total_params != EXPECTED_TOTAL_PARAMS:
    raise RuntimeError(
        f"Wrong CNN-Transformer loaded. Expected {EXPECTED_TOTAL_PARAMS:,} total parameters, "
        f"but loaded {total_params:,}. Do not use these results."
    )

print("\nPASS: Loaded the 613,889-parameter CNN-Transformer v2 improved model.")
model.summary()


Loading model: /content/drive/MyDrive/drone_audio_processed/model_cnntransformer_v2_improved.keras

Trying AttentionPooling variant: direct_weights
FAILED: direct_weights: ValueError: A total of 1 objects could not be loaded. Example error message for object <AttentionPoolingDirect name=attn_pool, built=True>:

Layer 'attn_pool' expected 2 variables, but received 0 variables during loading. Expected: ['kernel', 'bias']

List of objects that could not be loaded:
[<AttentionPoolingDirect name=attn_pool, built=True>]

Trying AttentionPooling variant: nested_dense_score
FAILED: nested_dense_score: ValueError: A total of 1 objects could not be loaded. Example error message for object <Dense name=score, built=True>:

Layer 'score' expected 2 variables, but received 0 variables during loading. Expected: ['kernel', 'bias']

List of objects that could not be loaded:
[<Dense name=score, built=True>]

Trying AttentionPooling variant: mean_fallback_diagnostic_only
Loaded with mean_fallback_diagnos

RuntimeError: Could not load the 613,889-parameter CNN-Transformer model with any AttentionPooling variant.

direct_weights: ValueError: A total of 1 objects could not be loaded. Example error message for object <AttentionPoolingDirect name=attn_pool, built=True>:

Layer 'attn_pool' expected 2 variables, but received 0 variables during loading. Expected: ['kernel', 'bias']

List of objects that could not be loaded:
[<AttentionPoolingDirect name=attn_pool, built=True>]

nested_dense_score: ValueError: A total of 1 objects could not be loaded. Example error message for object <Dense name=score, built=True>:

Layer 'score' expected 2 variables, but received 0 variables during loading. Expected: ['kernel', 'bias']

List of objects that could not be loaded:
[<Dense name=score, built=True>]

mean_fallback_diagnostic_only: loaded but wrong parameter count 878,849

## 3. Feature extraction helpers

In [ ]:
def get_target_frames(model):
    shape = model.input_shape
    if isinstance(shape, list):
        shape = shape[0]
    if len(shape) != 4:
        raise ValueError(f"Expected input shape (None, mel, frames, channel), got {shape}")
    if shape[1] != N_MELS:
        raise ValueError(f"Expected {N_MELS} mel bins, model expects {shape[1]}")
    return int(shape[2])

TARGET_FRAMES = get_target_frames(model)
print("Model target frames:", TARGET_FRAMES)


def waveform_to_logmel(y, sr=SR_TARGET, pre_emphasis=0.0):
    y = np.asarray(y, dtype=np.float32)

    # Basic normalisation to reduce amplitude scale differences.
    peak = np.max(np.abs(y)) + 1e-9
    y = y / peak

    if pre_emphasis and pre_emphasis > 0:
        if len(y) > 1:
            y = np.concatenate([[y[0]], y[1:] - pre_emphasis * y[:-1]])

    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_mels=N_MELS,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        fmax=FMAX,
        power=2.0,
    )
    mel_db = librosa.power_to_db(mel, ref=1.0, top_db=TOP_DB).astype(np.float32)
    return mel_db


def match_frames(mel, target_frames=TARGET_FRAMES):
    if mel.shape[1] == target_frames:
        return mel
    if mel.shape[1] > target_frames:
        return mel[:, :target_frames]
    return np.pad(mel, ((0, 0), (0, target_frames - mel.shape[1])), mode="edge")


def mel_to_model_input(mel):
    mel = match_frames(mel)
    # Per-window z-score normalisation.
    mel = (mel - np.mean(mel)) / (np.std(mel) + 1e-6)
    return mel[..., None].astype(np.float32)


def waveforms_to_batch(waves, pre_emphasis=0.0):
    return np.stack([
        mel_to_model_input(waveform_to_logmel(w, SR_TARGET, pre_emphasis=pre_emphasis))
        for w in waves
    ]).astype(np.float32)


def outputs_to_prob(raw):
    raw = np.asarray(raw, dtype=np.float32).ravel()
    if np.nanmin(raw) >= 0.0 and np.nanmax(raw) <= 1.0:
        return raw.astype(np.float32)
    return (1.0 / (1.0 + np.exp(-raw))).astype(np.float32)


def predict_waveforms(waves, pre_emphasis=0.0, batch_size=BATCH_SIZE, label=""):
    probs = []
    n = len(waves)
    for start in range(0, n, batch_size):
        batch_waves = waves[start:start + batch_size]
        X = waveforms_to_batch(batch_waves, pre_emphasis=pre_emphasis)
        raw = model.predict(X, batch_size=batch_size, verbose=0)
        probs.append(outputs_to_prob(raw))
        if start == 0 or start % (batch_size * 25) == 0:
            print(f"{label} {min(start + batch_size, n)}/{n}")
    return np.concatenate(probs).astype(np.float32)


def make_windows(y, sr=SR_TARGET, win_s=WIN_S, hop_s=HOP_S):
    y = np.asarray(y, dtype=np.float32)
    win = int(round(win_s * sr))
    hop = int(round(hop_s * sr))
    if len(y) <= win:
        out = np.zeros(win, dtype=np.float32)
        start = (win - len(y)) // 2
        out[start:start + len(y)] = y
        return [out]
    return [y[i:i + win] for i in range(0, len(y) - win + 1, hop)]


def aggregate_clip(window_probs, method=EXTERNAL_AGGREGATION):
    p = np.asarray(window_probs, dtype=np.float32)
    if method == "max":
        return float(np.max(p))
    if method == "mean":
        return float(np.mean(p))
    if method == "p95":
        return float(np.percentile(p, 95))
    raise ValueError(method)


## 4. Metrics helpers

In [ ]:
def binary_metrics(y_true, y_prob, threshold=0.50):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0
    )
    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = np.nan

    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy),
        "roc_auc": float(auc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def threshold_sweep(y_int, p_int, y_ext, p_ext, thresholds=THRESHOLDS):
    rows = []
    for t in thresholds:
        im = binary_metrics(y_int, p_int, t)
        em = binary_metrics(y_ext, p_ext, t)
        rows.append({
            "threshold": float(t),

            "internal_accuracy": im["accuracy"],
            "internal_roc_auc": im["roc_auc"],
            "internal_precision": im["precision"],
            "internal_recall": im["recall"],
            "internal_f1": im["f1"],
            "internal_tn": im["tn"],
            "internal_fp": im["fp"],
            "internal_fn": im["fn"],
            "internal_tp": im["tp"],

            "external_accuracy": em["accuracy"],
            "external_roc_auc": em["roc_auc"],
            "external_precision": em["precision"],
            "external_recall": em["recall"],
            "external_f1": em["f1"],
            "external_tn": em["tn"],
            "external_fp": em["fp"],
            "external_fn": em["fn"],
            "external_tp": em["tp"],
        })
    return pd.DataFrame(rows)


## 5. Load internal DADS test set

In [ ]:
WAVEFORM_CACHE = find_existing(WAVEFORM_CACHE_CANDIDATES)
print("Using waveform cache:", WAVEFORM_CACHE)

cache = np.load(WAVEFORM_CACHE)
print("Cache keys:", cache.files)

# Expected keys from the corrected CNN v3 / LWCNN pipeline.
X_test_wav = cache["X_test_wav"].astype(np.float32)
y_test = cache["y_test_wav"].astype(int)

print("Internal test waveforms:", X_test_wav.shape)
print("Internal labels:", pd.Series(y_test).value_counts().sort_index().to_dict())


## 6. Download and prepare external Svanström dataset

In [ ]:
EXT_ROOT = Path("/content/external_drone_dataset")
EXT_ROOT.mkdir(parents=True, exist_ok=True)

EXT_ZIP_URL = "https://codeload.github.com/DroneDetectionThesis/Drone-detection-dataset/zip/refs/heads/master"
EXT_ZIP_PATH = EXT_ROOT / "repo.zip"
EXT_EXTRACT_DIR = EXT_ROOT / "repo"

if not EXT_ZIP_PATH.exists():
    print("Downloading external dataset...")
    r = requests.get(EXT_ZIP_URL, stream=True)
    r.raise_for_status()
    with open(EXT_ZIP_PATH, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)

if not EXT_EXTRACT_DIR.exists() or not list(EXT_EXTRACT_DIR.glob("*")):
    print("Extracting external dataset...")
    with zipfile.ZipFile(EXT_ZIP_PATH, "r") as z:
        z.extractall(EXT_EXTRACT_DIR)

subdirs = [p for p in EXT_EXTRACT_DIR.glob("*") if p.is_dir()]
EXT_REPO_ROOT = subdirs[0] if subdirs else EXT_EXTRACT_DIR

external_wavs = sorted(glob.glob(str(EXT_REPO_ROOT / "**" / "*.wav"), recursive=True))
print("External wav files:", len(external_wavs))
print("First files:", [os.path.basename(p) for p in external_wavs[:10]])

def label_external_file(path):
    name = os.path.basename(path).lower()
    # Dataset has drone/background/helicopter files.
    return 1 if "drone" in name else 0

print("External class counts by filename:",
      pd.Series([label_external_file(p) for p in external_wavs]).value_counts().sort_index().to_dict())


## 7. Run recalculation for preprocessing variants

In [ ]:
all_variant_summaries = []
all_sweeps = {}

for variant_name, pre_emph in PREPROCESSING_VARIANTS.items():
    print("\n" + "=" * 90)
    print("PREPROCESSING VARIANT:", variant_name, "pre_emphasis =", pre_emph)
    print("=" * 90)

    variant_dir = OUTPUT_DIR / variant_name
    variant_dir.mkdir(parents=True, exist_ok=True)

    # Internal predictions
    internal_probs = predict_waveforms(
        X_test_wav,
        pre_emphasis=pre_emph,
        batch_size=BATCH_SIZE,
        label=f"Internal {variant_name}:"
    )

    internal_pred_df = pd.DataFrame({
        "sample_index": np.arange(len(y_test)),
        "y_true": y_test,
        "cnn_transformer_prob": internal_probs,
        "preprocessing_variant": variant_name,
        "model_path": str(MODEL_PATH),
    })
    internal_pred_path = variant_dir / "cnn_transformer_internal_predictions.csv"
    internal_pred_df.to_csv(internal_pred_path, index=False)

    # External clip predictions
    external_rows = []
    external_window_rows = []

    for clip_idx, wav_path in enumerate(external_wavs):
        y_clip = label_external_file(wav_path)
        y, sr = librosa.load(wav_path, sr=None, mono=True)
        if sr != SR_TARGET:
            y = librosa.resample(y, orig_sr=sr, target_sr=SR_TARGET)
            sr = SR_TARGET

        windows = make_windows(y, SR_TARGET)
        win_probs = predict_waveforms(
            np.asarray(windows, dtype=np.float32),
            pre_emphasis=pre_emph,
            batch_size=BATCH_SIZE,
            label=f"External {variant_name} clip {clip_idx+1}/{len(external_wavs)}:"
        )
        clip_prob = aggregate_clip(win_probs, EXTERNAL_AGGREGATION)

        external_rows.append({
            "clip_index": clip_idx,
            "filename": os.path.basename(wav_path),
            "file": wav_path,
            "y_true": y_clip,
            "n_windows": len(windows),
            "cnn_transformer_clip_prob": clip_prob,
            "aggregation": EXTERNAL_AGGREGATION,
            "preprocessing_variant": variant_name,
        })

        for j, p in enumerate(win_probs):
            external_window_rows.append({
                "clip_index": clip_idx,
                "window_index": j,
                "filename": os.path.basename(wav_path),
                "y_true": y_clip,
                "cnn_transformer_window_prob": float(p),
                "preprocessing_variant": variant_name,
            })

    external_df = pd.DataFrame(external_rows)
    external_window_df = pd.DataFrame(external_window_rows)

    external_clip_path = variant_dir / "cnn_transformer_external_clip_predictions.csv"
    external_window_path = variant_dir / "cnn_transformer_external_window_predictions.csv"
    external_df.to_csv(external_clip_path, index=False)
    external_window_df.to_csv(external_window_path, index=False)

    y_ext = external_df["y_true"].values.astype(int)
    p_ext = external_df["cnn_transformer_clip_prob"].values.astype(float)

    sweep_df = threshold_sweep(y_test, internal_probs, y_ext, p_ext, THRESHOLDS)
    sweep_df["preprocessing_variant"] = variant_name
    sweep_path = variant_dir / "cnn_transformer_threshold_sweep.csv"
    sweep_df.to_csv(sweep_path, index=False)
    all_sweeps[variant_name] = sweep_df

    m050 = binary_metrics(y_test, internal_probs, 0.50)
    best_row = sweep_df.sort_values(["external_f1", "external_recall"], ascending=False).iloc[0].to_dict()

    summary = {
        "preprocessing_variant": variant_name,
        "model_path": str(MODEL_PATH),
        "total_params": total_params,
        "trainable_params": trainable_params,
        "non_trainable_params": non_trainable_params,
        "internal_0p50_accuracy": m050["accuracy"],
        "internal_0p50_roc_auc": m050["roc_auc"],
        "internal_0p50_precision": m050["precision"],
        "internal_0p50_recall": m050["recall"],
        "internal_0p50_f1": m050["f1"],
        "internal_0p50_tn": m050["tn"],
        "internal_0p50_fp": m050["fp"],
        "internal_0p50_fn": m050["fn"],
        "internal_0p50_tp": m050["tp"],
        "best_external_f1_threshold": best_row["threshold"],
        "best_external_f1": best_row["external_f1"],
        "best_external_precision": best_row["external_precision"],
        "best_external_recall": best_row["external_recall"],
        "best_external_tn": best_row["external_tn"],
        "best_external_fp": best_row["external_fp"],
        "best_external_fn": best_row["external_fn"],
        "best_external_tp": best_row["external_tp"],
        "external_roc_auc": best_row["external_roc_auc"],
    }
    all_variant_summaries.append(summary)

    summary_df = pd.DataFrame([summary])
    summary_path = variant_dir / "cnn_transformer_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    print("\nInternal threshold 0.50:")
    print(json.dumps(m050, indent=2))
    print("\nBest external-F1 threshold:")
    print(json.dumps(best_row, indent=2))
    print("\nSaved:")
    print(" ", internal_pred_path)
    print(" ", external_clip_path)
    print(" ", external_window_path)
    print(" ", sweep_path)
    print(" ", summary_path)

summary_all_df = pd.DataFrame(all_variant_summaries)
summary_all_path = OUTPUT_DIR / "cnn_transformer_all_preprocessing_variant_summary.csv"
summary_all_df.to_csv(summary_all_path, index=False)

print("\n" + "=" * 90)
print("ALL VARIANT SUMMARY")
print("=" * 90)
display(summary_all_df)
print("Saved:", summary_all_path)


## 8. Create thesis-ready figures

In [ ]:
def row_percentages(cm):
    return cm / cm.sum(axis=1, keepdims=True) * 100


def plot_confusion_matrix(cm, title, out_path):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["No Drone", "Drone"])
    ax.set_yticklabels(["No Drone", "Drone"])

    labels = np.array([["TN", "FP"], ["FN", "TP"]])
    pct = row_percentages(cm)
    vmax = cm.max()

    for i in range(2):
        for j in range(2):
            colour = "white" if cm[i, j] > vmax * 0.45 else "black"
            ax.text(j, i - 0.18, labels[i, j], ha="center", va="center",
                    fontweight="bold", color=colour)
            ax.text(j, i + 0.08, f"{int(cm[i, j]):,}\n({pct[i, j]:.1f}%)",
                    ha="center", va="center", color=colour)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Count")
    fig.tight_layout()
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()


for variant_name in PREPROCESSING_VARIANTS.keys():
    variant_dir = OUTPUT_DIR / variant_name
    sweep_df = pd.read_csv(variant_dir / "cnn_transformer_threshold_sweep.csv")

    # Confusion matrices: threshold 0.50 and best external-F1 threshold.
    pred_df = pd.read_csv(variant_dir / "cnn_transformer_internal_predictions.csv")
    y_int = pred_df["y_true"].values.astype(int)
    p_int = pred_df["cnn_transformer_prob"].values.astype(float)

    best_t = float(sweep_df.sort_values(["external_f1", "external_recall"], ascending=False).iloc[0]["threshold"])

    for t, label in [(0.50, "0p50"), (best_t, f"{best_t:.2f}".replace(".", "p"))]:
        y_pred = (p_int >= t).astype(int)
        cm = confusion_matrix(y_int, y_pred, labels=[0, 1])
        out_path = variant_dir / f"cnn_transformer_internal_confusion_threshold_{label}.png"
        plot_confusion_matrix(
            cm,
            f"CNN-Transformer Internal Test Set\n{variant_name}, threshold = {t:.2f}",
            out_path
        )
        print("Saved:", out_path)

    # Threshold sweep figure.
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(sweep_df["threshold"], sweep_df["internal_f1"], marker="o", label="Internal drone F1")
    ax.plot(sweep_df["threshold"], sweep_df["external_recall"], marker="s", label="External drone recall")
    ax.plot(sweep_df["threshold"], sweep_df["external_f1"], marker="^", label="External drone F1")
    ax.axvline(best_t, linestyle=":", label=f"Best external F1 = {best_t:.2f}")
    ax.set_xlabel("Decision threshold")
    ax.set_ylabel("Score")
    ax.set_title(f"CNN-Transformer Threshold Sweep ({variant_name})")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    sweep_fig_path = variant_dir / "cnn_transformer_threshold_sweep.png"
    fig.savefig(sweep_fig_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", sweep_fig_path)


## 9. Report wording

In [ ]:
summary_all_df = pd.read_csv(OUTPUT_DIR / "cnn_transformer_all_preprocessing_variant_summary.csv")
display(summary_all_df)

print("\nChoose the preprocessing variant that matches your training/evaluation pipeline.")
print("If one variant reproduces your known internal AUC/F1 values, use that variant for the thesis.")
print("\nReport wording for each variant:\n")

for _, row in summary_all_df.iterrows():
    variant = row["preprocessing_variant"]
    print("=" * 90)
    print("Variant:", variant)
    print("=" * 90)

    print(
        f"On the internal test set, the CNN-Transformer achieved an accuracy of "
        f"{100*row['internal_0p50_accuracy']:.2f}%, a ROC-AUC of {row['internal_0p50_roc_auc']:.4f}, "
        f"a drone-class precision of {100*row['internal_0p50_precision']:.2f}%, "
        f"a drone-class recall of {100*row['internal_0p50_recall']:.2f}%, and a drone-class F1 score of "
        f"{row['internal_0p50_f1']:.4f} at a decision threshold of 0.50. "
        f"The internal confusion matrix contained {int(row['internal_0p50_tn']):,} true negatives, "
        f"{int(row['internal_0p50_fp']):,} false positives, {int(row['internal_0p50_fn']):,} false negatives, "
        f"and {int(row['internal_0p50_tp']):,} true positives."
    )
    print()
    print(
        f"Based on external drone F1, the optimal CNN-Transformer threshold was "
        f"{row['best_external_f1_threshold']:.2f}. At this threshold, the model achieved "
        f"external precision of {100*row['best_external_precision']:.2f}%, "
        f"external recall of {100*row['best_external_recall']:.2f}%, and external F1 of "
        f"{row['best_external_f1']:.4f}."
    )
    print()
    print(
        f"The final CNN-Transformer model contained {int(row['total_params']):,} total parameters, "
        f"of which {int(row['trainable_params']):,} were trainable and "
        f"{int(row['non_trainable_params']):,} were non-trainable."
    )
    print()
